In [ ]:
import s3fs
import rasterio
import xarray as xr
import rioxarray
import duckdb
import numpy as np
import matplotlib.pyplot as plt
import boto3
import pandas as pd
import geopandas as gpd

## Check that treemap is in scratch bucket

In [ ]:
fs = s3fs.S3FileSystem()
fs.ls("s3://nasa-cryo-scratch/s-kganz/treemap/Data/")

## Summarize BA per treemap ID and damage-causing agent

In [ ]:
# It seems like downloading the database is the best option. Save to temp so we
# don't have extra data lying around.
fs.download(
    "nasa-cryo-scratch/s-kganz/treemap/Data/TreeMap2016_tree_table.db",
    "/tmp/TreeMap2016_tree_table.db"
)

In [ ]:
# Species codes that match each damage agent.
# From 10.1016/j.foreco.2025.122549
host_spcodes = {
    "mtn_pb": (113,108,122,101,116,117,119,109,102,142,114,133,104,105,106),
    "fir_eng": (15,20,17,22),
    "pin_ips": (106, 133),
    "spr_bet": (93, 96, 97),
    "west_pb": (122,109,112,117,116,101),
    "west_bb": (19, 18),
    "doug_fb": (202,),
    "jeff_pb": (116, 109, 137)
}

db = duckdb.connect("/tmp/TreeMap2016_tree_table.db")
db.sql("SHOW TABLES")

In [ ]:
db.sql("PRAGMA table_info(TreeMap2016_tree_table)")

In [ ]:
db.sql("SELECT * FROM TreeMap2016_tree_table LIMIT 5").df()

In [ ]:
# pi/4 * DIA (in.) * DIA (in.) * TPA (1/ac) = BA (in^2/ac) 
# BA (in^2) * pi/4 (-) * 6.4516e-4 (m^2/in^2) * 247.105 (ac/km^2) = BA (m^2/km^2)
ba_conversion_factor = 6.4516e-4 * 247.105 * np.pi / 4
print(ba_conversion_factor)

In [ ]:
# Query will omit TM IDs that no basal area because of dead trees or lack
# of host. So we first have to make a destination table with zeros.
all_hostba = db.sql(
    """
    SELECT DISTINCT tm_id FROM TreeMap2016_tree_table
    """
).df()
all_hostba = all_hostba.set_index("tm_id")
print(all_hostba.shape)

In [ ]:
# Then iterate through each damage-causing agent, query, and assign values
# to TM IDs that have nonzero basal area.
ba_by_dca = []
for (dca, spcodes) in host_spcodes.items():
    query = f'''
    SELECT 
        tm_id,
        SUM(DIA * DIA * TPA_UNADJ * {ba_conversion_factor}) as {dca}
    FROM TreeMap2016_tree_table
    WHERE 
        STATUSCD == 1 AND
        SPCD IN {spcodes}
    GROUP BY tm_id
    HAVING {dca} > 0
    '''
    #print(query)
    hostba_where_present = db.sql(query).df()
    hostba_where_present = hostba_where_present.set_index("tm_id")
    ba_by_dca.append(hostba_where_present)

In [ ]:
all_hostba = pd.concat([all_hostba] + ba_by_dca, axis=1).fillna(0)

In [ ]:
# Account for nodata pixels
nodata_tmid = 2147483647
all_hostba.loc[nodata_tmid] = 0
all_hostba.tail()

## Open treemap from scratch bucket

In [ ]:
session = rasterio.session.AWSSession(boto3.Session(), requester_pays=True)

chunksize_x = 2880
chunksize_y = 2880
treemap = rioxarray.open_rasterio(
    "s3://nasa-cryo-scratch/s-kganz/treemap/Data/TreeMap2016.tif", 
    band_as_variable=True,
    chunks=dict(x=chunksize_x, y=chunksize_y)
)
print(treemap.rio.crs)
treemap

In [ ]:
# Figure out processing extent
usfs_regions = gpd.read_file("../data_in/usfs_region_boundaries/usfs_regions_simple.shp").to_crs(treemap.rio.crs)
usfs_regions_explode = usfs_regions.geometry.explode()
usfs_regions_explode = usfs_regions_explode[usfs_regions_explode.geometry.area > 2e11]
bounds = usfs_regions_explode.total_bounds
xmin, ymin, xmax, ymax = bounds
print(bounds)

In [ ]:
# sel() must be exactly on the beginning/end of a chunk for ease of use
# with map_blocks(). So snap to the nearest coordinate and then snap
# to the nearest chunk
x_snap = treemap.x.sel(x=[xmin, xmax], method="nearest")
y_snap = treemap.y.sel(y=[ymin, ymax], method="nearest")
x_idx = np.where(treemap.x.isin(x_snap))[0]
y_idx = np.where(treemap.y.isin(y_snap))[0]

print("Before snapping:", x_idx, y_idx)

x_idx[0] = int(chunksize_x * (np.floor(x_idx[0] / chunksize_x)))
x_idx[1] = int(chunksize_x * (np.ceil(x_idx[1] / chunksize_x)))
y_idx[0] = int(chunksize_y * (np.floor(y_idx[0] / chunksize_y)))
y_idx[1] = int(chunksize_y * (np.ceil(y_idx[1] / chunksize_y)))

print("After snapping:", x_idx, y_idx)

In [ ]:
treemap_clip = treemap.isel(
    x=slice(*x_idx),
    y=slice(*y_idx) 
)
treemap_clip

In [ ]:
# Assert that all chunks are the same size.
for dim in treemap_clip.chunksizes:
    sizes = np.array(treemap_clip.chunksizes[dim])
    assert (sizes == sizes[0]).all()

## Try processing a block

The goal here is to replace each pixel in TreeMap with the corresponding entry in the above table. There are many more treemap IDs in the above table than unique values in each block, so it would be very efficient to np.where() for everything. Instead we use fast Pandas indexing with .loc[] to transfer the relevant rows of the table to the pixel. Then to fit everything in memory we coarsen to ~900 m pixels.

In [ ]:
%%time
block = treemap_clip.isel(x=slice(38_000, 40_000), y=slice(38_000, 40_000)).compute()
out_shape = block.band_1.shape + (all_hostba.shape[1],)
block_tmids = block.band_1.data.flatten()
block_ba = xr.DataArray(
    data=all_hostba.loc[block_tmids].to_numpy().reshape(out_shape),
    dims=block.band_1.dims + ("dca",),
    coords=block.coords
)
block_coarse = block_ba.coarsen(x=10, y=10).mean()
block_coarse = block_coarse.assign_coords(dca=all_hostba.columns)

## Process using map_blocks()

The flatten step pulls all the data in the chunk into memory, so the best option for processing the full array is `xr.map_blocks`.

In [ ]:
# Make a local cluster for parallelism
from dask.distributed import Client, LocalCluster
from dask.diagnostics import ProgressBar

#cluster = LocalCluster(n_workers=3, memory_limit="12GiB")
#client = cluster.get_client()
#client

In [ ]:
coarsen_factor = 10

def process_block(block: xr.Dataset) -> xr.DataArray:
    block_tmids = block.band_1.data.flatten()
    block_ba = xr.DataArray(
        data=all_hostba.loc[block_tmids].to_numpy().reshape(block.band_1.shape + (all_hostba.shape[1],)),
        dims=block.band_1.dims + ("dca",),
        coords=dict(dca=all_hostba.columns, **block.coords)
    )
    block_coarse = block_ba.coarsen(dict(y=coarsen_factor, x=coarsen_factor), boundary="trim").mean()
    return block_coarse

In [ ]:
template = treemap_clip.band_1.coarsen(x=coarsen_factor, y=coarsen_factor, boundary="trim").mean()
template = template.expand_dims(dca=all_hostba.columns, axis=-1)
template

In [ ]:
mpb_ba = xr.map_blocks(
    func=process_block,
    obj=treemap_clip,
    template=template
)

In [ ]:
with ProgressBar():
    mpb_ba = mpb_ba.compute()

In [ ]:
mpb_ba

In [ ]:
mpb_ba.to_zarr("../data_working/treemap2016_hostba.zarr")

In [ ]:
# Smaller array for plotting
ba_coarse = mpb_ba.coarsen(x=8, y=8).mean()

In [ ]:
from cartopy import crs as ccrs
from cartopy import feature as cfeature

source_proj = ccrs.AlbersEqualArea(
    central_latitude=23,
    central_longitude=-96,
    standard_parallels=(29.5, 45.5)
)
target_proj = ccrs.Mercator()

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(12, 6), subplot_kw=dict(projection=target_proj))
xmin, ymin, xmax, ymax = mpb_ba.rio.transform_bounds(3857)

for dca, ax in zip(all_hostba.columns, axes.flat):
    ba_coarse.sel(dca=dca).plot(ax=ax, transform=source_proj, add_colorbar=False, add_labels=False, xlim=[xmin, xmax], ylim=[ymin, ymax])
    ax.set_title(dca)
    ax.coastlines()
    ax.add_feature(cfeature.STATES)
    
plt.show()